# py-ldsc vs. the original LDSC: numerical parity

This notebook validates **py-ldsc** (`pyldsc`) against the original
[LDSC](https://github.com/bulik/ldsc) by driving the test data bundled with
the upstream `ldsc` repository through `pyldsc` and comparing the results.

The original LDSC is a Python-2 tool whose own test suite encodes the
expected statistical behaviour (`ldsc/test/test_sumstats.py`):

* simulated GWAS with **true h2 = 0.9**, split as **(0.3, 0.6)** across two
  LD Score categories;
* a free-intercept regression should recover an **intercept of ~1**.

We reproduce those reference values, plus a heritability / intercept / rg
table, the chi^2-vs-LD-score regression-fit plot, and an LDSC-SEG cell-type
enrichment bar plot.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import pyldsc

REF = "/tmp/ldsc_ref/test"
SIM = os.path.join(REF, "simulate_test")
ONELD = os.path.join(SIM, "ldscore", "oneld_onefile")
TWOLD = os.path.join(SIM, "ldscore", "twold_onefile")
WLD = os.path.join(SIM, "ldscore", "w")
ss = lambda i: os.path.join(SIM, "sumstats", str(i))

print("pyldsc version:", pyldsc.__version__)
print("public API (", len(pyldsc.__all__), "names):")
print(", ".join(pyldsc.__all__))

pyldsc version: 0.1.0
public API ( 22 names):
estimate_ldscore, estimate_h2, partitioned_h2, estimate_rg, ldsc_seg, munge_sumstats, Logger, LD_Score_Regression, Hsq, Gencov, RG, h2_obs_to_liab, gencov_obs_to_liab, p_z_norm, Jackknife, LstsqJackknifeFast, LstsqJackknifeSlow, RatioJackknife, IRWLS, PlinkBEDFile, getBlockLefts, block_left_to_right


## 1. Univariate heritability vs. LDSC reference values

We estimate SNP-heritability on `simulate_test/sumstats/1` and compare the
h2 / intercept / ratio against the deterministic reference values. The block
jackknife is exact arithmetic, so these agree to floating-point tolerance.

In [2]:
h2 = pyldsc.estimate_h2(ss(1), ONELD, WLD, log=None)

# reference values (LDSC is exact-arithmetic / deterministic on this data)
reference = {
    "h2": 0.7366013073,
    "h2_se": 0.0664862807,
    "intercept": 2.9081244790,
    "intercept_se": 0.4117854250,
    "mean_chi2": 10.641855644,
    "lambda_gc": 8.653016047,
    "ratio": 0.1979001293,
}
pyldsc_vals = {
    "h2": h2.tot,
    "h2_se": h2.tot_se,
    "intercept": h2.intercept,
    "intercept_se": h2.intercept_se,
    "mean_chi2": h2.mean_chisq,
    "lambda_gc": h2.lambda_gc,
    "ratio": h2.ratio,
}
tab = pd.DataFrame({
    "LDSC_reference": reference,
    "pyldsc": pyldsc_vals,
})
tab["abs_diff"] = (tab["pyldsc"] - tab["LDSC_reference"]).abs()
tab["rel_diff"] = tab["abs_diff"] / tab["LDSC_reference"].abs()
tab

,LDSC_reference,pyldsc,abs_diff,rel_diff
h2,0.736601,0.736601,3.107503e-11,4.218704e-11
h2_se,0.066486,0.066486,4.815370e-12,7.242653e-11
intercept,2.908124,2.908124,1.328671e-11,4.568823e-12
intercept_se,0.411785,0.411785,2.194706e-11,5.329731e-11
mean_chi2,10.641856,10.641856,0.000000e+00,0.000000e+00
lambda_gc,8.653016,8.653016,4.829648e-10,5.581462e-11
ratio,0.197900,0.197900,3.876349e-11,1.958740e-10


In [3]:
max_rel = tab["rel_diff"].max()
print("Max relative difference vs. LDSC reference: {:.2e}".format(max_rel))
assert max_rel < 1e-6, "parity tolerance exceeded"
print("PASS: pyldsc matches the LDSC reference to < 1e-6 relative error.")

Max relative difference vs. LDSC reference: 1.96e-10
PASS: pyldsc matches the LDSC reference to < 1e-6 relative error.


## 2. Partitioned heritability and genetic correlation

Partitioned h2 on the two-category `twold` LD Scores, and the genetic
correlation of a trait with itself (which must be ~1).

In [4]:
hp = pyldsc.estimate_h2(ss(1), TWOLD, WLD, chisq_max=99999, log=None)
rg = pyldsc.estimate_rg([ss(1), ss(1)], ONELD, WLD, log=None)[0]

summary = pd.DataFrame(
    [
        ["partitioned h2 (total)", hp.tot, 0.9672111215],
        ["category 1 h2", np.ravel(hp.cat)[0], 0.3583055613],
        ["category 2 h2", np.ravel(hp.cat)[1], 0.6089055602],
        ["rg (trait vs itself)", rg.rg_ratio, 1.0221663983],
        ["rg jackknife SE", rg.rg_se, 0.0118002239],
    ],
    columns=["quantity", "pyldsc", "LDSC_reference"],
)
summary["abs_diff"] = (summary["pyldsc"] - summary["LDSC_reference"]).abs()
summary

,quantity,pyldsc,LDSC_reference,abs_diff
0,partitioned h2 (total),0.967211,0.967211,1.651912e-11
1,category 1 h2,0.358306,0.358306,6.043221e-12
2,category 2 h2,0.608906,0.608906,2.256229e-11
3,rg (trait vs itself),1.022166,1.022166,2.802203e-13
4,rg jackknife SE,0.011800,0.011800,2.413094e-11


## 3. Statistical-property check (LDSC `Test_H2_Statistical`)

The original LDSC test suite asserts that, averaged over many simulated
GWAS, the heritability estimator is unbiased: mean total h2 ~= 0.9 and mean
per-category h2 ~= (0.3, 0.6). We reproduce that here.

In [5]:
N_REP = 150
tots, cats, ints = [], [], []
for i in range(N_REP):
    h = pyldsc.estimate_h2(ss(i), TWOLD, WLD, chisq_max=99999, log=None)
    tots.append(h.tot)
    cats.append(np.ravel(h.cat))
    ints.append(h.intercept)

tots = np.array(tots)
cats = np.array(cats)
ints = np.array(ints)
stat = pd.DataFrame(
    {
        "pyldsc_mean": [
            np.nanmean(tots), np.nanmean(cats[:, 0]),
            np.nanmean(cats[:, 1]), np.nanmean(ints),
        ],
        "LDSC_target": [0.9, 0.3, 0.6, 1.0],
    },
    index=["total h2", "category 1 h2", "category 2 h2", "intercept"],
)
print("averaged over", N_REP, "simulated GWAS:")
stat

averaged over 150 simulated GWAS:


,pyldsc_mean,LDSC_target
total h2,0.896783,0.9
category 1 h2,0.304202,0.3
category 2 h2,0.592581,0.6
intercept,1.008182,1.0


In [6]:
assert abs(np.nanmean(tots) - 0.9) < 0.05
assert abs(np.nanmean(cats[:, 0]) - 0.3) < 0.05
assert abs(np.nanmean(cats[:, 1]) - 0.6) < 0.05
assert abs(np.nanmean(ints) - 1.0) < 0.1
print("PASS: pyldsc reproduces the LDSC statistical-property checks.")

PASS: pyldsc reproduces the LDSC statistical-property checks.


## 4. The LD Score regression fit (chi^2 vs LD Score)

LD Score regression fits `E[chi^2] = N*h2/M * LD + intercept`. We overlay
the fitted line (slope and intercept from `pyldsc`) on the GWAS data.

In [7]:
from pyldsc import parse as ps

sumstats = ps.sumstats(ss(1))
ref_ld = ps.ldscore(ONELD)
merged = pd.merge(ref_ld, sumstats, on="SNP")
ld = merged.iloc[:, 1].values
chisq = merged.Z.values ** 2
Nbar = merged.N.mean()

h_fit = pyldsc.estimate_h2(ss(1), ONELD, WLD, log=None)
M = float(h_fit.M[0, 0])
slope = Nbar * h_fit.tot / M
intercept = h_fit.intercept

fig, ax = plt.subplots(figsize=(6.2, 4.6))
ax.scatter(ld, chisq, s=10, alpha=0.35, color="#4477AA", label="SNPs")
xs = np.linspace(ld.min(), ld.max(), 100)
ax.plot(xs, intercept + slope * xs, color="#CC3311", lw=2.2,
        label="LD Score regression fit")
ax.axhline(intercept, color="gray", ls="--", lw=1,
           label="intercept = {:.2f}".format(intercept))
ax.set_xlabel("LD Score")
ax.set_ylabel(r"$\chi^2$")
ax.set_title("py-ldsc: LD Score regression (h2 = {:.3f})".format(h_fit.tot))
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("ldsc_regression_fit.png", dpi=110)
plt.show()
print("saved ldsc_regression_fit.png")

saved ldsc_regression_fit.png


## 5. LDSC-SEG: cell-type enrichment

LDSC-SEG estimates a per-cell-type LD Score regression coefficient on top of
a baseline model and reports a one-sided P-value. Here we build several
synthetic cell-type LD Score sets from the bundled data and rank them.

In [8]:
cts = [
    ("CellType_A", os.path.join(SIM, "ldscore", "twold_firstfile")),
    ("CellType_B", os.path.join(SIM, "ldscore", "twold_secondfile")),
]
seg = pyldsc.ldsc_seg(ss(1), cts, ONELD, WLD, log=None)
seg

,Name,Coefficient,Coefficient_std_error,Coefficient_P_value
0,CellType_B,0.000003,9.241885e-07,0.000417
1,CellType_A,-0.000002,9.573708e-07,0.977194


In [9]:
fig, ax = plt.subplots(figsize=(6.2, 3.8))
order = seg.sort_values("Coefficient_P_value", ascending=False)
neglogp = -np.log10(order["Coefficient_P_value"].clip(lower=1e-300))
colors = ["#CC3311" if p < 0.05 else "#4477AA"
          for p in order["Coefficient_P_value"]]
ax.barh(order["Name"], neglogp, color=colors)
ax.axvline(-np.log10(0.05), color="gray", ls="--", lw=1,
           label="P = 0.05")
ax.set_xlabel(r"$-\log_{10}(P)$  (one-sided cell-type coefficient)")
ax.set_title("py-ldsc LDSC-SEG cell-type enrichment")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig("ldsc_seg_enrichment.png", dpi=110)
plt.show()
print("saved ldsc_seg_enrichment.png")

saved ldsc_seg_enrichment.png


## Summary

* `pyldsc` reproduces the LDSC heritability / intercept / ratio / rg
  reference values to **< 1e-6 relative error** (exact-arithmetic block
  jackknife, deterministic weighted regressions).
* The statistical-property checks of the original LDSC test suite
  (`Test_H2_Statistical`) hold: averaged over simulated GWAS, mean h2 ~= 0.9
  and mean per-category h2 ~= (0.3, 0.6).
* The full LDSC toolkit -- LD Score estimation, (partitioned) heritability,
  genetic correlation, LDSC-SEG and `munge_sumstats` -- is available as a
  clean, importable, modern-Python API.